In [ ]:
%config InteractiveShell.cache_size = 0
%load_ext autoreload
%autoreload 2
%matplotlib inline


In [ ]:
! aws s3 sync s3://narang99-private/lucent-workdir/images ./image-shards

In [ ]:
Path("flat-images").mkdir(parents=True, exist_ok=True)
from olt.df_to_ir import extract_images_to_flat_folder

extract_images_to_flat_folder(Path("image-shards"), Path("flat-images"))

In [ ]:
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from collections import defaultdict


base_report_dir = Path("mass-train-reports")


In [ ]:
# use when you want to clear jupyter output cache memory, this is very annoying
# i would rather not have this, i dont see a single reason on why they would want to do this
# i like the convenience of not writing print of display, wtf.
%reset out

In [ ]:
# we do tabby.
DOWNLOAD_LABEL = 18
DEST_DIR = Path("18")


! aws s3 cp s3://narang99-private/lucent-workdir/images/{DOWNLOAD_LABEL}/000000.tar \
    imagenet-label-{DOWNLOAD_LABEL}-000000.tar

! rm -rf {DEST_DIR}
! mkdir -p {DEST_DIR}
! tar -xf imagenet-label-{DOWNLOAD_LABEL}-000000.tar -C {DEST_DIR}

In [ ]:
# now what? hehe, imma forget. we also want the report yes, idk why i downloaded the data
# copy main neuron's report
! cp ~/Downloads/lucent-reports/mixed4e_1x1_pre_relu_conv-55/overlay-reports-hyperparameter-grid-search/norm_l2-pca_None-min_cluster_size_20-min_samples_5-method_leaf/report.csv \
        ./this-and-prev/mixed4e1x1_report.csv

In [ ]:
# thankfully, we have layer_name and channel in these reports. 
# i can merge all the reports in 1 yayyy
# huhuhuhuhuhuhuhhuhuhu

import pandas as pd
df = pd.read_csv("this-and-prev/mixed4e1x1_report.csv")

df.head()

# download reports and extract

In [ ]:
# ill need to download all the reports sadly, thats a lot of data (csvs are inside tgz files)
! aws s3 sync s3://narang99-private/lucent-workdir/mass-train-reports/ this-and-prev/mass-train-reports/

In [ ]:
# extract reports now
for layer_dir in list(base_report_dir.glob("*")):
    if not layer_dir.is_dir():
        continue
    for report_tgz in tqdm(list(layer_dir.glob("*.tgz")), desc=layer_dir.name):
        ! cd {layer_dir} && tar -xf {report_tgz.name}

# make main df

In [ ]:
! pwd

In [ ]:
csv_files = list(base_report_dir.rglob("report.csv"))
csv_files[:10]

In [ ]:
dfs = []
for f in tqdm(csv_files):
    df = pd.read_csv(f)
    df = df[df.cluster_label != -1].reset_index()
    dfs.append(df)

In [ ]:
dfs.append(pd.read_csv("mixed4e1x1_report.csv"))
df = pd.concat(dfs)
df = df[df.cluster_label  != -1]

# data for each cluster label

For each cluster label, I'm now going to get all the parent layer-chan-cluster-label combos.  
We want to analyse how each cluster looks. It would be useful to also add some more terms:

- Actual activation value, pointwise value.
- image key

with these,i can also guess how much and how each neuron+label combo contributes to the final activation.  
Some sort of visualisation might be handy, but for now, just collecting data is fine, we'll also look at each case separately, trying to come up with smartypants visualisations is hurting right now.   

In [ ]:
df.head()

In [ ]:
def get_deps_df_for_single_cluster_label(df, child_layer_name, child_channel, cluster_label_in_child_neuron):
    parent_by_count = defaultdict(lambda: 0)
    child_df = df[
        (df.layer_name == child_layer_name) & 
        (df.channel == child_channel) & 
        (df.cluster_label == cluster_label_in_child_neuron)
    ]
    image_keys = child_df.input_image_key.unique()
    # df = df.set_index('input_image_key')
    # df = df.sort_index()
    for image_key in tqdm(image_keys):
        image_keys_df = df[df.input_image_key == image_key]
        child_image_keys_df = image_keys_df[image_keys_df.layer_name == child_layer_name]
        for row in child_image_keys_df.itertuples():
            y, x = row.y_position, row.x_position
            parents_image_keys_df = image_keys_df[
                (image_keys_df.layer_name != child_layer_name) &
                (image_keys_df.y_position == y) &
                (image_keys_df.x_position == x)
            ]
            for tup in parents_image_keys_df.itertuples():
                parent_key = (tup.layer_name, tup.channel, tup.cluster_label)
                parent_by_count[parent_key] += 1
    return parent_by_count



def crop_top(image_path, max_height):
    """
    Crops an image to at most `max_height` pixels from the top.
    If the image is already shorter than max_height, it's left unchanged.
    """
    img = Image.open(image_path)
    width, height = img.size

    if height <= max_height:
        # Image is already smaller (or equal) — no cropping needed
        cropped = img
    else:
        # box = (left, upper, right, lower)
        cropped = img.crop((0, 0, width, max_height))

    return cropped


In [ ]:
# we get the snapshot for our current input image key
# but we dont have all the inputs right now shiz, it would take a long time to download
# maybe working in colab is the correct thing to do
# hmmmmmmmm, this is getting painful yes, ill also need to backup the reports somewhere for fast access
# maybe backup the flat images dir to s3 too. thats a lot of work. for now, lets just then, do basic shit
# we would in the end, need layer-name, channel, cluster-label, position, 
# technically, im doing a lot of looping and shit, do i need to?
# actually yea, tis def slightly complicated. it would have been very useful to get activations hmpffffffffffff.  
# kya kar sakte hai. agar activation leta to bi bhot saare activations aate na, kese unko structure karta? 
# generally, we would assume their values to have the same sign at least. but they can wildly change. lets think about that later then


# we now have the function to extract top parts of the jpeg images which contain the cluster reports. 
# with a given base dir combo, i can easily now extract these. and create a new index html

In [ ]:
def get_for_child(df, layer_name, channel, cluster_label):
    parent_by_count = get_deps_df_for_single_cluster_label(
        df, layer_name, channel, cluster_label
    )

    cluster_parents_df = pd.DataFrame(
        [
            (layer, channel, cluster_label, count)
            for (layer, channel, cluster_label), count in parent_by_count.items()
        ],
        columns=["layer", "channel", "cluster_label", "count"],
    )
    return cluster_parents_df



In [ ]:
crp = crop_top("/Users/hariomnarang/Desktop/personal/hiccup-ide/olt/notebooks/this-and-prev/mass-train-reports/mixed4d_1x1_pre_relu_conv/5/cluster_3_combined.jpeg",  470)
display(crp)

In [ ]:
cluster_53_parents_df = get_for_child(df, "mixed4e_1x1_pre_relu_conv", 55, 53)
cluster_53_parents_df.sort_values("count", ascending=False).head(20)


In [ ]:
cluster_31_parents_df = get_for_child("mixed4e_1x1_pre_relu_conv", 55, 31)
cluster_31_parents_df.sort_values("count", ascending=False).head(20)

In [ ]:
cluster_53_parents_df.head()

In [ ]:
len(unique_neurons)

In [ ]:
len(cluster_31_parents_df[["layer", "channel"]].drop_duplicates())

In [ ]:
# 107 are polysemantic across examples. Note that we are not catching this many neurons in each run
# which is concerning. i dont have enough data to do like "great" data analysis lol. 
# now there is the other fact that i would like to miss the outliers.

polysemantic_between_31_and_53

In [ ]:
# now cluster 31 and cluster 53 mei, konse neurons are common?
# sabse pehle vo dekhege

polysemantic_between_31_and_53 = 0

unique_neurons = cluster_53_parents_df[["layer", "channel"]].drop_duplicates()
for tup in unique_neurons.itertuples():
    one_neuron = cluster_31_parents_df[
        (cluster_31_parents_df.layer == tup.layer) &
        (cluster_31_parents_df.channel == tup.channel)
    ]
    if len(one_neuron) > 1:
        polysemantic_between_31_and_53 += 1
        print(cluster_53_parents_df[(cluster_53_parents_df.layer == tup.layer) & (cluster_53_parents_df.channel == tup.channel)])
        print("----------------------------------------------------------------------------------------------------")
        print(one_neuron)
        print("####################################################################################################")
        # print(f"{one_neuron.layer}:{one_neuron.channel} c53:{tup.count} ")
    

In [ ]:
one_neuron["count"].sum()

In [ ]:
# we want to visualise this information
# we will compare this with the PWs in the report to begin with.



cluster_53_parents_df

In [ ]:
from lucent.modelzoo import inceptionv1

device = "cpu"
model = inceptionv1(pretrained=True)
model = model.to(device)
model = model.eval()


plt.style.use("dark_background")

In [ ]:
df[
    (df.imagenet_label == 18) & 
    (df.layer_name == "mixed4e_1x1_pre_relu_conv") & 
    (df.cluster_label == 53) &
    (df.input_image_key == "c7a212ed40dd844091632efe7e6d5ecb799131bc3ffdcc2e1d5a61516d7fdcda")
]

In [ ]:
# ohk, first get for all previous layers ka output for pos 6,6 we get heatmaps
layer_names = ['mixed4d_1x1_pre_relu_conv',
       'mixed4d_3x3_pre_relu_conv', 
       'mixed4d_5x5_pre_relu_conv', 'mixed4d_pool_reduce_pre_relu_conv',]

stacked_frequencies = []

for name in layer_names:
    chans = model.get_submodule(name).weight.shape[0]
    # ndl = NeuronDeepLift(model, model.get_submodule(name))
    for chan in tqdm(list(range(chans)), desc=name):
        print(name, chan)
        ndf = cluster_53_parents_df[
            (cluster_53_parents_df.layer == name) &
            (cluster_53_parents_df.channel == chan)
        ]
        print(ndf)
        print("---------------------------------------------------------")
        # this would have rows, what do we do with this? for now, add the counts xD
        stacked_frequencies.append(ndf["count"].sum())
stacked_frequencies = np.array(stacked_frequencies)

In [ ]:
# ohk, ive got the images now
# basically get the avg of all patches
# and avg of all pws
# for df.cluster-label == 53 and our neuron

In [ ]:
ourdf = df[
    (df.layer_name == "mixed4e_1x1_pre_relu_conv") & 
    (df.cluster_label == 53) &
    (df.channel == 55)
]

w = model.get_submodule("mixed4e_1x1_pre_relu_conv").weight[55].detach().cpu().reshape(-1)


patches, pws = [], []

for ik  in tqdm(ourdf.input_image_key.unique()):
    timg = transform(Image.open(Path("flat-images") / f"{ik}.jpeg"))[None]
    acts = InputOutputModelSnapshot.get_activations(timg, model, ["mixed4e_1x1_pre_relu_conv"])
    for tup in ourdf[ourdf.input_image_key == ik].itertuples():
        
        ip = acts["mixed4e_1x1_pre_relu_conv"]["input"][0, :, tup.y_position, tup.x_position]
        patches.append(ip)
        pws.append(w*ip)
        

avg_patch = torch.stack(patches).mean(dim=0)
avg_pw = torch.stack(pws).mean(dim=0)

In [ ]:
avg_pw.reshape(22,24)[17,20], avg_pw.reshape(22,24)[17,7]

In [ ]:
acts["mixed4e_1x1_pre_relu_conv"]["output"][0, 55, 10, 7]

In [ ]:
sf = stacked_frequencies.copy()
sf[avg_patch.numpy() == 0] = 0

In [ ]:
plt.plot(torch.cumsum(torch.sort(avg_pw).values, dim=0))

In [ ]:
plt.plot(sorted(avg_pw))

In [ ]:
# mixed4d_3x3_pre_relu_conv/271/index.html
cluster_53_parents_df[
    (cluster_53_parents_df.layer == "mixed4d_3x3_pre_relu_conv") &
    (cluster_53_parents_df.channel == 271)
]

In [ ]:
# mixed4d_3x3_pre_relu_conv/271/index.html
cluster_53_parents_df[
    (cluster_53_parents_df.layer == "mixed4d_5x5_pre_relu_conv") &
    (cluster_53_parents_df.channel == 28)
]

In [ ]:
avg_pw[383], avg_pw[428]

In [ ]:
torch.argmin(avg_pw)

In [ ]:
# 17,7

avg_pw.reshape(22,24)[17,7], avg_pw.reshape(22,24)[15,23], avg_pw.reshape(22,24)[17,20]

In [ ]:
# not good enough, lets get the averages then
from olt.show import show_single_channel_red_green_black as S
S([
    stacked_frequencies.reshape(22,24),
    avg_patch.reshape(22,24),
    avg_pw.reshape(22,24),
], 10, 3, viztype="local")

In [ ]:
# now cluster 31 and cluster 53 mei, konse neurons are common?
# sabse pehle vo dekhege
# now we would also like to know which ones are very less "firing" (sabse simple  hai ke ek hi sample mei fire karre)

polysemantic_between_31_and_53 = 0

unique_neurons = cluster_53_parents_df[["layer", "channel"]].drop_duplicates()
for tup in unique_neurons.itertuples():
    one_neuron = cluster_31_parents_df[
        (cluster_31_parents_df.layer == tup.layer) &
        (cluster_31_parents_df.channel == tup.channel)
    ]
    if len(one_neuron) > 1:
        main_neuron = cluster_53_parents_df[
            (cluster_53_parents_df.layer == tup.layer) & (cluster_53_parents_df.channel == tup.channel)
        ]
        
        
        total_count = main_neuron["count"].sum() + one_neuron["count"].sum()
        threshold = np.floor(total_count * 0.05)
        main_neuron = main_neuron[main_neuron["count"] > threshold]
        one_neuron = one_neuron[one_neuron["count"] > threshold]


        if len(main_neuron) > 0 and len(one_neuron) > 0:
            print(main_neuron)
            print("----------------------------------------------------------------------------------------------------")
            print(one_neuron)
    
            print("####################################################################################################")
            # print(f"{one_neuron.layer}:{one_neuron.channel} c53:{tup.count} ")
            
            polysemantic_between_31_and_53 += 1


In [ ]:
cluster_41_parents_df = get_for_child("mixed4e_1x1_pre_relu_conv", 55, 41)
cluster_41_parents_df.sort_values("count", ascending=False).head(20)

In [ ]:
cluster_24_parents_df = get_for_child("mixed4e_1x1_pre_relu_conv", 55, 24)
cluster_24_parents_df.sort_values("count", ascending=False).head(20)

In [ ]:
cluster_61_parents_df = get_for_child("mixed4e_1x1_pre_relu_conv", 55, 61)
cluster_61_parents_df.sort_values("count", ascending=False).head(20)

In [ ]:
cluster_59_parents_df = get_for_child("mixed4e_1x1_pre_relu_conv", 55, 59)
cluster_59_parents_df.sort_values("count", ascending=False).head(20)

In [ ]:
# we have the locations and everything, we just need to go to the base-reports-dir and go ahead. 
# btw, this new fucntion for cropping would be very useful for generating data heheheheheeee
# to put in report. imma stupid? yes. ;_;. should have done that before.  

In [ ]:

# child_by_parents = defaultdict(set)

parent_by_count = defaultdict(lambda: 0)
# child_key = ("mixed4e_1x1_pre_relu_conv", 55, 61)
for image_key in tqdm(idf.input_image_key.unique()):
    uidf = idf[idf.input_image_key == image_key]
    our_layer_only_tabby_df = uidf[(uidf.layer_name == "mixed4e_1x1_pre_relu_conv") & (uidf.cluster_label == 61)]
    for row in our_layer_only_tabby_df.itertuples():
        y, x = row.y_position, row.x_position
        # we want to set this
        # get the ones for positions not in our layer, all parents
        parent_df = uidf[(uidf.x_position == x) & (uidf.y_position == y) & (uidf.layer_name != "mixed4e_1x1_pre_relu_conv")]
        for tup in parent_df.itertuples():
            parent_key = (tup.layer_name, tup.channel, tup.cluster_label)
            parent_by_count[parent_key] += 1
            # child_by_parents[child_key].add()
        
            
        # child_by_parents[("mixed4e_1x1_pre_relu_conv", 55, 61)] 

# activation ranges


for a given cluster, find the range of incoming patch values, and the range of pointwise multiplications.   
We do the animal leg cluster first. cluster 53.   


Go through all images in this. for each position of 53 cluster, get the pointwise multiplication, and th epatch too.  
We have the code for this already.  


First I'll focus on the PW ranges I guess.  


I cant find anything lol. Should i now check what the activation output ranges are for different clusters?   
This is getting out of hand. its not giving a lot of useful information at all. hmpf.   


inter-layer interaction is hard to understand. the first problem is not having enough data.   
HDBSCAN is missing stuff cuz its not doing the range thing.   


to find a good correlation stat, we would want to make sure everything is "caught" too. but that is not the case.    


Think again. We are assuming that there are "activation ranges" for each concept. The ranges together with other neurons, decide if someone downstream is going to fire.  
How does someone downstream decide to fire? the range it outputs is consistent? Not for max values btw.   


Now, say i create a pattern in my weights which maximally aligns with one pattern, the highest pattern out there.  My range would be big now. How can someone downstream handle or normalise that big range?

- they could just accept it and then give a big range themselves (let the problem solve itself downstream, or maybe not? in this case, that range would keep expanding or something).
- decrease the range by attaching a smaller weight. if there are other features which have more predictable range and we can assign a higher weight to them, then stuff is "balanced".


Both seem reasonable strategies.  In the end, we have a GAP, exploded range would end up giving highs. in this case, the exploded range is also a signal.    

The first thing we want then, is to see the range of activations in neurons before us. and compare them to the range of pws in our neuron.  
And then compare them with the range in the detected category.   

Is this the right thing to do? In the end, it really is just a multiply and a sum, if there is a predictable input pattern, the neuron would emit a predictable output pattern. depending on the sparisty of the weights of course (which decides which dimensions get priority). Note that a smaller weight at one place does not necessarily mean a small value in that place. PW is the main source of truth in the end).  


So what is the variance of the PW of the neuron's output, wrt others?   
Now again, this is hard to put in words.   


In the end, we would like to see the variance of each dimension in the final sum.   
what do we do once we have the variance though. Aah shit wait, if data is incomplete, analysis might confound me.   

In [ ]:
ourdf = df[
    (df.layer_name == "mixed4e_1x1_pre_relu_conv") & 
    (df.cluster_label == 53) &
    (df.channel == 55)
]

w = model.get_submodule("mixed4e_1x1_pre_relu_conv").weight[55].detach().cpu().reshape(-1)


patches, pws = [], []

for ik  in tqdm(ourdf.input_image_key.unique()):
    timg = transform(Image.open(Path("flat-images") / f"{ik}.jpeg"))[None]
    acts = InputOutputModelSnapshot.get_activations(timg, model, ["mixed4e_1x1_pre_relu_conv"])
    for tup in ourdf[ourdf.input_image_key == ik].itertuples():        
        ip = acts["mixed4e_1x1_pre_relu_conv"]["input"][0, :, tup.y_position, tup.x_position]
        patches.append(ip)
        pws.append(w*ip)
        

patches = torch.stack(patches)
pws = torch.stack(pws)

In [ ]:
pws.shape

In [ ]:
_, axes = plt.subplots(48, 11, figsize=(11*3, 48*3), sharey=True)
axes = axes.flatten()


xs = list(range(len(pws[:,0])))
for i in range(len(axes)):
    axes[i].scatter(
        xs, pws[:, i]
    )

In [ ]:
%reset out

In [ ]:
# now, lets look at the correlation between each input location and the final sum.  
# how do we do that? the simplest way is to actually plot the incoming dim as x, and the sum as y

pws.shape

In [ ]:
xs.shape, ys.shape

In [ ]:
def _plot_one_pw(pws, i, ax1):
    ys = pws.sum(dim=1)
    xs = pws[:, i]
    
    idxs = torch.argsort(xs)
    xs = xs[idxs]
    ys = ys[idxs]
    
    ax1.plot(xs, ys)
    ax2 = ax1.twinx()
    ax2.hist(xs, color="red", alpha=0.3)
    ax2.set_yticks([])  # optional: hide the histogram's y-axis ticks since it's just a density cue

In [ ]:
_, axes = plt.subplots(48, 11, figsize=(11*3, 48*3), sharey=True)
axes = axes.flatten()


for i in range(len(axes)):
    _plot_one_pw(pws, i, axes[i])
plt.show()

In [ ]:
from sklearn.preprocessing import normalize
_, axes = plt.subplots(48, 11, figsize=(11*3, 48*3), sharey=True)
axes = axes.flatten()

npws = torch.tensor(normalize(pws, "l2"))
for i in range(len(axes)):
    _plot_one_pw(npws, i, axes[i])
plt.show()

In [ ]:
fig, ax1 = plt.subplots()

ax1.plot(xs, ys)

ax2 = ax1.twinx()
ax2.hist(xs, color="red", alpha=0.3)
ax2.set_yticks([])  # optional: hide the histogram's y-axis ticks since it's just a density cue

In [ ]:
plt.plot(xs, ys)
# plt.scatter(range(len(xs)), xs, color="red")
plt.hist(xs, color="red")
plt.show()

In [ ]:
ys = pws.sum(dim=1)
xs = pws[:, 0]

idxs = torch.argsort(xs)
xs = xs[idxs]
ys = ys[idxs]

_, axes = plt.subplots(1, 2, figsize=(6,3))
axes[0].plot(xs, ys, )
axes[1].hist(xs)
plt.show()

In [ ]:
_, axes = plt.subplots(48, 11, figsize=(11*3, 48*3), sharey=True)
axes = axes.flatten()


ys = pws.sum(dim=1)
for i in range(len(axes)):
    xs = pws[:, i]
    axes[i].scatter(
        xs, ys
    )
plt.show()

In [ ]:
# plt.scatter(range(len(pws)), pws[:, 428], color="red")
plt.scatter(range(len(patches)), pws[:, 428], color="blue")
plt.show()

In [ ]:
plt.scatter(range(len(patches)), pws[:, 383], color="blue")
plt.show()

# what next? 

we have a df containing all the data now. its useful to look at a single image's data first, we filter by that input key.  
Then for a given point, i would like to get the receptive field of that point, then get the points inside that receptive field in the df.  
This gives us the labels, easy.  

In [ ]:
from PIL import Image
image_key = "901e389eb7629a7266d79eccba5679381fe2eec4586c18f8581b8fef56a583c9"
pil = Image.open(f"this-and-prev/tabby_cat/{image_key}.jpg")
pil

In [ ]:
all_cats = list(Path("this-and-prev/tabby_cat").glob("*.jpg"))
# name by counts of found objects
for c in all_cats:
    print(c.stem, len(df[df.input_image_key == c.stem]))
all_cat_keys = [a.stem for a in all_cats]

In [ ]:
all_cat_keys = set(all_cat_keys)

In [ ]:
idf = df[df.input_image_key.isin(all_cat_keys)].reset_index()
print("len", len(idf))
idf.head()

In [ ]:
from collections import defaultdict

# child_by_parents = defaultdict(set)

parent_by_count = defaultdict(lambda: 0)
# child_key = ("mixed4e_1x1_pre_relu_conv", 55, 61)
for image_key in tqdm(idf.input_image_key.unique()):
    uidf = idf[idf.input_image_key == image_key]
    our_layer_only_tabby_df = uidf[(uidf.layer_name == "mixed4e_1x1_pre_relu_conv") & (uidf.cluster_label == 61)]
    for row in our_layer_only_tabby_df.itertuples():
        y, x = row.y_position, row.x_position
        # we want to set this
        # get the ones for positions not in our layer, all parents
        parent_df = uidf[(uidf.x_position == x) & (uidf.y_position == y) & (uidf.layer_name != "mixed4e_1x1_pre_relu_conv")]
        for tup in parent_df.itertuples():
            parent_key = (tup.layer_name, tup.channel, tup.cluster_label)
            parent_by_count[parent_key] += 1
            # child_by_parents[child_key].add()
        
            
        # child_by_parents[("mixed4e_1x1_pre_relu_conv", 55, 61)] 

In [ ]:
child_by_parents

In [ ]:
import pandas as pd

clus61_parents_df = pd.DataFrame(
    [(layer, channel, cluster_label, count) for (layer, channel, cluster_label), count in parent_by_count.items()],
    columns=["layer", "channel", "cluster_label", "count"]
)

In [ ]:
clus61_parents_df.to_csv("this-and-prev/cluster_61_parents.csv")
clus61_parents_df

In [ ]:
uidf[(uidf.layer_name == "mixed4e_1x1_pre_relu_conv") & (uidf.cluster_label == 61)]

In [ ]:
row = idf[idf.layer_name == "mixed4e_1x1_pre_relu_conv"].iloc[0]
y, x = row.y_position, row.x_position

In [ ]:
y,x

In [ ]:
# label is 61 in mixed4e_1x1_pre_relu_conv
# we find the unique set for each found 61
# we are for now positive that these are very "mouth" centric clusters


idf[(idf.cluster_label == 61) & (idf.layer_name == "mixed4e_1x1_pre_relu_conv")]

In [ ]:
for row in idf[(idf.y_position == y) & (idf.x_position == x)].itertuples():
    print(f"{row.layer_name}/{row.channel}/index.html")
    print("\tcluster label", row.cluster_label)
    # print(row.layer_name, row.channel, row.cluster_label)

In [ ]:
idf.head()

## Tabby refactored

In [ ]:
def _get_neuron_mask(df):
    return (df.layer_name == "mixed4e_1x1_pre_relu_conv") & (df.channel == 55)

TABBY_CLUSTER_ID = 61
neuron_mask = (df.layer_name == "mixed4e_1x1_pre_relu_conv") & (df.channel == 55)
tabby_mask = (df.cluster_label == TABBY_CLUSTER_ID)
# first get the ones which were found
idf = df[_get_neuron_mask(df) & tabby_mask]
# then get a df from the main df with input image keys of the ones found above
idf = df[df.input_image_key.isin(idf.input_image_key)].reset_index()
print(len(idf))
idf.head()

In [ ]:
df[df.input_image_key == "901e389eb7629a7266d79eccba5679381fe2eec4586c18f8581b8fef56a583c9"]

In [ ]:
df[(df.layer_name == "mixed4e_1x1_pre_relu_conv") & (df.cluster_label == 61)]

In [ ]:
# im collecting counts here
# its easy and simple,

from collections import defaultdict

CLUSTER_ID = TABBY_CLUSTER_ID

image_by_count = defaultdict(lambda: 0)
for image_key in tqdm(idf.input_image_key.unique()):
    uidf = idf[idf.input_image_key == image_key]
    our_neuron_only_df = uidf[_get_neuron_mask(uidf) & (uidf.cluster_label == CLUSTER_ID)]
        
    # our_layer_only_tabby_df = uidf[(uidf.layer_name == "mixed4e_1x1_pre_relu_conv") & (uidf.cluster_label == 61)]

    for row in our_neuron_only_df.itertuples():
        y, x = row.y_position, row.x_position
        parent_df = uidf[(uidf.x_position == x) & (uidf.y_position == y)]
        parent_df = parent_df[(~_get_neuron_mask(parent_df))]
        image_by_count[(image_key, y, x)] = len(parent_df)
        # we want to set this
        # get the ones for positions not in our layer, all parents
        
        # for tup in parent_df.itertuples():
        #     parent_key = (tup.layer_name, tup.channel, tup.cluster_label)
        #     parent_by_count[parent_key] += 1
            # child_by_parents[child_key].add()
        
            
        # child_by_parents[("mixed4e_1x1_pre_relu_conv", 55, 61)] 

In [ ]:
image_by_count

In [ ]:
image_key = "901e389eb7629a7266d79eccba5679381fe2eec4586c18f8581b8fef56a583c9"
image_key_df = df[df.input_image_key == image_key]
image_key_df[_get_neuron_mask(image_key_df)]

In [ ]:
for y, x in [[9,8], [9,10], [10,8]]:
    print(y, x, len(image_key_df[(image_key_df.y_position == y) & (image_key_df.x_position == x)]))

In [ ]:
from olt.tfms import transform, inverse_transform
from olt.html_report import make_overlay_heatmap



pil = Image.open(f"this-and-prev/tabby_cat/{image_key}.jpg")
timg = transform(pil)[None]
pil

In [ ]:
from captum.attr import NeuronDeepLift, NeuronIntegratedGradients

selector_by_hm = {}

image_key_df_6_6 = image_key_df[(image_key_df.y_position == 8) & (image_key_df.x_position == 8)]

for row in tqdm(image_key_df_6_6.itertuples(), total=len(image_key_df_6_6)):
    ndl = NeuronDeepLift(model, model.get_submodule(row.layer_name))
    neuron_selector = (row.channel, row.y_position, row.x_position)
    attr = ndl.attribute(timg, neuron_selector)
    hm = make_overlay_heatmap(model, row.layer_name, neuron_selector, pil)
    key = (row.layer_name, *neuron_selector)
    selector_by_hm[key] = hm

In [ ]:
from olt.show import show_grid
from PIL import ImageOps

imgs, titles = [], []
for k,v in selector_by_hm.items():
    imgs.append(v)
    titles.append("\n".join([str(_k) for _k in k]))


# # imgs = apply_score_border_and_brightness(imgs, list(range(-len(imgs) // 2, len(imgs)//2)))

# # 7,11

# show_grid(imgs, 7, 11, titles)
# plt.show()

In [ ]:
show_grid(imgs, 5, 6, titles)
plt.show()

In [ ]:
image_key_df[(image_key_df.y_position == 9) & (image_key_df.x_position == 8)]

In [ ]:
image_key = "f0e1ad69f7d5edcaa867b16f538baa0abc28a9119e37176852f88c6722bc5c0b"
image_key_df = df[df.input_image_key == image_key]
image_key_df[_get_neuron_mask(image_key_df)]

In [ ]:
for y, x in [[6,4], [6,6], [6,8]]:
    print(y, x, len(image_key_df[(image_key_df.y_position == y) & (image_key_df.x_position == x)]))

In [ ]:
from olt.tfms import transform, inverse_transform
from olt.html_report import make_overlay_heatmap



pil = Image.open(f"this-and-prev/tiger_cat/{image_key}.jpg")
timg = transform(pil)[None]

In [ ]:
image_key_df[(image_key_df.y_position == 6) & (image_key_df.x_position == 6)]

In [ ]:
from lucent.modelzoo import inceptionv1

device = "cpu"
model = inceptionv1(pretrained=True)
model = model.to(device)
model = model.eval()


plt.style.use("dark_background")

In [ ]:
image_key_df.layer_name.unique()

In [ ]:
layer_by_chan_by_hm["mixed4d_3x3_pre_relu_conv"][5]

In [ ]:
# the vis below, is the most boring thing ever
# we need to now visualise the whole 528 grid, not very painful i think
# all we need to do is get the position on all previous ones for this image,
# then we would also like to assign a score to each
# and the grid should have some sort of alpha blending


def apply_score_brightness(images, scores, min_brightness=0.1):
    scores = np.array(scores, dtype=float)
    s_min, s_max = scores.min(), scores.max()
    
    if s_max == s_min:
        normalized = np.ones_like(scores)
    else:
        normalized = (scores - s_min) / (s_max - s_min)
    
    # Scale to [min_brightness, 1.0]
    factors = min_brightness + (1.0 - min_brightness) * normalized
    
    result = []
    for img, factor in zip(images, factors):
        arr = np.array(img).astype(float)
        arr = np.clip(arr * factor, 0, 255).astype(np.uint8)
        result.append(Image.fromarray(arr))
    
    return result

def apply_score_border_and_brightness(images, scores, border_width=10, min_brightness=0.2):
    scores = np.array(scores, dtype=float)
    
    # Step 1: add full-bright border based on sign
    bordered = []
    for img, score in zip(images, scores):
        color = (0, 255, 0) if score >= 0 else (255, 0, 0)
        if score == 0:
            color = (0,0,0)
        bordered.append(ImageOps.expand(img, border=border_width, fill=color))
    
    # Step 2: scale brightness of everything together
    return apply_score_brightness(bordered, np.abs(scores), min_brightness=min_brightness)

In [ ]:
# ohk, first get for all previous layers ka output for pos 6,6 we get heatmaps
y_pos, x_pos = 6, 6
layer_names = ['mixed4d_1x1_pre_relu_conv',
       'mixed4d_3x3_pre_relu_conv', 'mixed4d_pool_reduce_pre_relu_conv',
       'mixed4d_5x5_pre_relu_conv']

layer_by_chan_by_hm = defaultdict(dict)

for name in layer_names:
    chans = model.get_submodule(name).weight.shape[0]
    ndl = NeuronDeepLift(model, model.get_submodule(name))
    for chan in tqdm(list(range(chans)), desc=name):
        neuron_selector = (chan, y_pos, x_pos)
        attr = ndl.attribute(timg, neuron_selector)
        # print("making overlay with", name, neuron_selector)
        hm = make_overlay_heatmap(model, name, neuron_selector, pil)

        layer_by_chan_by_hm[name][chan] = hm

In [ ]:
# flattened now

flattened_hms = []
layer_order = ["mixed4d_1x1_pre_relu_conv", "mixed4d_3x3_pre_relu_conv", "mixed4d_5x5_pre_relu_conv", "mixed4d_pool_reduce_pre_relu_conv"]


flattened_chan_by_layer_name_and_chan = {}

for layer_name in layer_order:
    chans = model.get_submodule(layer_name).weight.shape[0]
    for c in range(chans):
        flattened_hms.append(layer_by_chan_by_hm[layer_name][c])
        flattened_chan_by_layer_name_and_chan[len(flattened_hms) - 1] = (layer_name, c)
        # layer_name_and_chan_by_flattened[] = len(flattened_hms) - 1

len(flattened_hms)

In [ ]:
from olt.act import InputOutputModelSnapshot

acts = InputOutputModelSnapshot.get_activations(timg, model, ["mixed4e_1x1_pre_relu_conv"])

In [ ]:
# 'mixed4d_1x1_pre_relu_conv', 0
image_key_df[
    (image_key_df.layer_name == "mixed4d_1x1_pre_relu_conv") & 
    (image_key_df.channel == 0) & 
    (image_key_df.x_position == x_pos) & 
    (image_key_df.y_position == y_pos)
]

In [ ]:
score_1_if_found = []

for chan in range(len(flattened_hms)):
    layer_name, orig_chan = flattened_chan_by_layer_name_and_chan[chan]
    filtered = image_key_df[
        (image_key_df.layer_name == layer_name) & 
        (image_key_df.channel == orig_chan) & 
        (image_key_df.x_position == x_pos) & 
        (image_key_df.y_position == y_pos)
    ]
    if len(filtered) == 0:
        score_1_if_found.append(0)
    else:
        score_1_if_found.append(pw[chan])

In [ ]:
image_key_df[image_key_df.channel == 104]

In [ ]:
from olt.show import show_single_channel_red_green_black as S
this_act = acts["mixed4e_1x1_pre_relu_conv"]["input"][0, :, 6, 6]
layer_weight = model.get_submodule("mixed4e_1x1_pre_relu_conv").weight[55].reshape(-1).detach().cpu()
pw = layer_weight * this_act 
S([
    this_act.reshape(48,11), pw.numpy().reshape(48,11)], viztype="local"
)

In [ ]:
%reset out

In [ ]:
# make the maximum 0, its dwarfing the others
score_1_if_found[493] = 0.
imgs = apply_score_border_and_brightness(flattened_hms, score_1_if_found, 5, min_brightness=0.1)
show_grid(imgs, 48, 11)
plt.savefig("./only-found.png")
plt.close()

In [ ]:
# make the maximum 0, its dwarfing the others
npw = pw.clone()
npw[493] = 0.
imgs = apply_score_border_and_brightness(flattened_hms, npw, 5, min_brightness=0.1)
show_grid(imgs, 48, 11)
plt.savefig("./all-tabys-pw.png")
plt.close()

In [ ]:
imgs = apply_score_border_and_brightness(flattened_hms, pw, 5, min_brightness=0.1)
show_grid(imgs, 48, 11)
plt.show()

In [ ]:
from captum.attr import NeuronDeepLift, NeuronIntegratedGradients

selector_by_hm = {}

image_key_df_6_6 = image_key_df[(image_key_df.y_position == 6) & (image_key_df.x_position == 6)]

for row in tqdm(image_key_df_6_6.itertuples(), total=len(image_key_df_6_6)):
    ndl = NeuronIntegratedGradients(model, model.get_submodule(row.layer_name))
    neuron_selector = (row.channel, row.y_position, row.x_position)
    attr = ndl.attribute(timg, neuron_selector, n_steps=64)
    hm = make_overlay_heatmap(model, row.layer_name, neuron_selector, pil)
    key = (row.layer_name, *neuron_selector)
    selector_by_hm[key] = hm

In [ ]:
image_key_df_6_6[image_key_df_6_6.layer_name == "mixed4e_1x1_pre_relu_conv"]

In [ ]:
make_overlay_heatmap(model, "mixed4e_1x1_pre_relu_conv", [55,6,6], pil)

In [ ]:
from olt.show import show_grid
from PIL import ImageOps

imgs, titles = [], []
for k,v in selector_by_hm.items():
    imgs.append(v)
    titles.append("\n".join([str(_k) for _k in k]))


# imgs = apply_score_border_and_brightness(imgs, list(range(-len(imgs) // 2, len(imgs)//2)))

# 7,11

show_grid(imgs, 7, 11, titles)
plt.show()

## now onto cars

label in our neuron 59

In [ ]:
df[df.layer_name == "mixed4e_1x1_pre_relu_conv"].cluster_label.value_counts()

In [ ]:
def _get_neuron_mask(df):
    return (df.layer_name == "mixed4e_1x1_pre_relu_conv") & (df.channel == 55)

neuron_mask = (df.layer_name == "mixed4e_1x1_pre_relu_conv") & (df.channel == 55)
car_mask = (df.cluster_label == 59)
# first get the ones which were found
idf = df[_get_neuron_mask(df) & car_mask]
# then get a df from the main df with input image keys of the ones found above
idf = df[df.input_image_key.isin(idf.input_image_key)].reset_index()
print(len(idf))
idf.head()

In [ ]:
idf.input_image_key.nunique()

In [ ]:
from collections import defaultdict


parent_by_count = defaultdict(lambda: 0)
for image_key in tqdm(idf.input_image_key.unique()):
    uidf = idf[idf.input_image_key == image_key]
    our_neuron_only_df = uidf[_get_neuron_mask(uidf)]
        
    # our_layer_only_tabby_df = uidf[(uidf.layer_name == "mixed4e_1x1_pre_relu_conv") & (uidf.cluster_label == 61)]

    for row in our_neuron_only_df.itertuples():
        y, x = row.y_position, row.x_position
        # we want to set this
        # get the ones for positions not in our layer, all parents
        parent_df = uidf[(uidf.x_position == x) & (uidf.y_position == y)]
        parent_df = parent_df[(~_get_neuron_mask(parent_df))]
        for tup in parent_df.itertuples():
            parent_key = (tup.layer_name, tup.channel, tup.cluster_label)
            parent_by_count[parent_key] += 1
            # child_by_parents[child_key].add()
        
            
        # child_by_parents[("mixed4e_1x1_pre_relu_conv", 55, 61)] 

In [ ]:
parent_by_count

# human skin check

cluster 41 has skin, lets see usme, i need to see what red means.  
For cat, the story is too weird or simple, im not sure. it does raise a lot of questions.   

For now, every neuron is shouting the same thing, it seems like a pass through. If it is a pass through though, then we do expect outputs to have same activations for a given cat category (or at least, be able to see finer clusters after clustering outputs).    



In [ ]:

def _get_neuron_mask(df):
    return (df.layer_name == "mixed4e_1x1_pre_relu_conv") & (df.channel == 55)

neuron_mask = (df.layer_name == "mixed4e_1x1_pre_relu_conv") & (df.channel == 55)
skin_mask = (df.cluster_label == 41)
# first get the ones which were found
idf = df[_get_neuron_mask(df) & skin_mask]
# then get a df from the main df with input image keys of the ones found above
idf = df[df.input_image_key.isin(idf.input_image_key)].reset_index()
print(len(idf))
idf.head()

In [ ]:
# im collecting counts here
# its easy and simple,

from collections import defaultdict

CLUSTER_ID = 41

image_by_count = defaultdict(lambda: 0)
for image_key in tqdm(idf.input_image_key.unique()):
    uidf = idf[idf.input_image_key == image_key]
    our_neuron_only_df = uidf[_get_neuron_mask(uidf) & (uidf.cluster_label == CLUSTER_ID)]
        
    # our_layer_only_tabby_df = uidf[(uidf.layer_name == "mixed4e_1x1_pre_relu_conv") & (uidf.cluster_label == 61)]

    for row in our_neuron_only_df.itertuples():
        y, x = row.y_position, row.x_position
        parent_df = uidf[(uidf.x_position == x) & (uidf.y_position == y)]
        parent_df = parent_df[(~_get_neuron_mask(parent_df))]
        image_by_count[(image_key, y, x)] = len(parent_df)
        # we want to set this
        # get the ones for positions not in our layer, all parents
        
        # for tup in parent_df.itertuples():
        #     parent_key = (tup.layer_name, tup.channel, tup.cluster_label)
        #     parent_by_count[parent_key] += 1
            # child_by_parents[child_key].add()
        
            
        # child_by_parents[("mixed4e_1x1_pre_relu_conv", 55, 61)] 

In [ ]:
image_by_count

In [ ]:
df[df.input_image_key == "cfbc2db49ab129f7fe489a07001bd86d99883c609c3e2d2559f829ff0f453730"]

In [ ]:
df[
    (df.input_image_key == "cfbc2db49ab129f7fe489a07001bd86d99883c609c3e2d2559f829ff0f453730") &
    (df.y_position == 9) &
    (df.x_position == 8)
][["layer_name", "channel", "cluster_label"]]

In [ ]:
pil = Image.open("this-and-prev/pinwheel/cfbc2db49ab129f7fe489a07001bd86d99883c609c3e2d2559f829ff0f453730.jpg")
timg = transform(pil)[None]

In [ ]:
image_key_df = df[df.input_image_key == "cfbc2db49ab129f7fe489a07001bd86d99883c609c3e2d2559f829ff0f453730"]

In [ ]:
from captum.attr import NeuronDeepLift, NeuronIntegratedGradients

selector_by_hm = {}

image_key_df_6_6 = image_key_df[(image_key_df.y_position == 9) & (image_key_df.x_position == 8)]

for row in tqdm(image_key_df_6_6.itertuples(), total=len(image_key_df_6_6)):
    ndl = NeuronDeepLift(model, model.get_submodule(row.layer_name))
    neuron_selector = (row.channel, row.y_position, row.x_position)
    attr = ndl.attribute(timg, neuron_selector)
    hm = make_overlay_heatmap(model, row.layer_name, neuron_selector, pil)
    key = (row.layer_name, neuron_selector[0], row.cluster_label)
    selector_by_hm[key] = hm

In [ ]:
from olt.show import show_grid
from PIL import ImageOps

imgs, titles = [], []
for k,v in selector_by_hm.items():
    imgs.append(v)
    titles.append("\n".join([str(_k) for _k in k]))


# imgs = apply_score_border_and_brightness(imgs, list(range(-len(imgs) // 2, len(imgs)//2)))

# 7,11

show_grid(imgs, 2, 7, titles)
plt.show()